# This notebook can be executed after getRegionalBox.ipynb

In [ ]:
#read("/Users/nobuaki/Library/Jupyter/kernels/julia-8-threads-1.12/kernel.json", String)

In [1]:
#ENV["JULIA_NUM_THREADS"]
Threads.nthreads()

1

In [ ]:
# Locate flexOPT securely without relying on @__DIR__ (unreliable in IJulia).
# If this notebook is outside the repository, set ENV["FLEXOPT_ROOT"] first.
import Pkg

function find_flexopt_root(start_dir=pwd())
    candidates = String[]
    if haskey(ENV, "FLEXOPT_ROOT")
        push!(candidates, abspath(expanduser(ENV["FLEXOPT_ROOT"])))
    end
    directory = abspath(start_dir)
    while true
        push!(candidates, directory)
        parent = dirname(directory)
        parent == directory && break
        directory = parent
    end
    for candidate in unique(candidates)
        project_file = joinpath(candidate, "Project.toml")
        source_dir = joinpath(candidate, "src")
        if isfile(project_file) && isfile(joinpath(source_dir, "commonBatchs.jl"))
            return candidate
        end
    end
    error("Cannot locate flexOPT. Start Jupyter inside the repository or set ENV[\"FLEXOPT_ROOT\"] to its absolute path.")
end

flexopt_root = find_flexopt_root()
Pkg.activate(flexopt_root)
@show VERSION Threads.nthreads() Base.active_project()
\n# Metal must be loaded before batchGPU.jl selects the backend.
using Metal
Metal.functional() || error("Metal.jl is loaded, but cannot access the Apple GPU")
@show Metal.devices()

#Pkg.resolve()
#Pkg.instantiate()
#Pkg.precompile()

# batchGPU should be at this level (I have not made it as a module yet, since the choice of Metal/CUDA should be done in a manual way)
include(joinpath(flexopt_root, "src", "batchFiles", "batchGPU.jl"))


include(joinpath(flexopt_root, "src", "commonBatchs.jl"))

include(joinpath(flexopt_root, "src", "planet1D.jl"))
planet1D.configure_input!()
include(joinpath(flexopt_root, "src", "GeoPoints.jl"))

using .commonBatchs, .planet1D, .GeoPoints\n

## ici, je dois recharger flexOPT pour débugger

In [ ]:
include(joinpath(flexopt_root, "src", "flexOPT.jl"))
using .flexOPT

# give me a model of material variables (like seismic models)

In [ ]:
modelName="marmousi"
imageFile="../dataInput/model/random/marmousi.png"
modelDefinitionMethod="2DimageFile" # ToyModel or 2DimageFile (or 1DsphericalPlanet)
model=defineModel(imageFile);
#
#boxGridsMarmousi = constructLocalBox(model,-3000.0,0.0,0.0,9200.0)
boxGrids = lazyProduceOrLoad("MarmousiCoordInfo",constructLocalBox,model,-3000.0,0.0,0.0,9200.0)
#seismicModelMarmousi = makeAdHocSeismicModel(model, 1.0, 2.8, 1.5, 5.5, 0.0, 3.2)
seismicModelMarmousi=lazyProduceOrLoad("seismicModelMarmousi",makeAdHocSeismicModel,model, 1.0, 2.8, 1.5, 5.5, 0.0, 3.2);

#constructLocalBox for marmousi models should be written!

In [ ]:
# what you need as a model:

#boxGrids = lazyProduceOrLoad("MarmousiCoordInfo"); # for a big model, I do not recommend to store these data

#seismicModelMarmousi=lazyProduceOrLoad("seismicModelMarmousi");

# choose your physics and get the semi symbolic OPT expression(s) 

In [ ]:
famousEquationType="2DacousticTime"
@show myEquationInside=famousEquations(famousEquationType)


In [ ]:
# Spatial grid spacing for the numerical model.
# The time step is chosen automatically below from a CFL target after the velocity model is loaded.
Δspace = (1.e2, 1.e2)
cfl_safety = 0.45

# Fallback only; overwritten after models[1] is defined.
Δ = (Δspace..., 1.0)

# or, if you want to force a manual value later:
# Δ = (1.e3, 1.e3, 0.05)


In [ ]:
models=[]
models=push!(models,seismicModelMarmousi.Vpv.*1.e3)

# Choose a stable-ish time step before constructing optRec.
# For 2D acoustic explicit-style marching, v * dt / min(dx, dz) should be below ~0.5-0.7.
finite_velocity_for_cfl = filter(isfinite, vec(Float64.(models[1])))
vmax_for_cfl = maximum(abs, finite_velocity_for_cfl)
dxmin_for_cfl = minimum(Float64.(Δspace))

cfl_info_preconstruction = (
    v_min = minimum(finite_velocity_for_cfl),
    v_max = maximum(finite_velocity_for_cfl),
    dxmin = dxmin_for_cfl,
    cfl_target = cfl_safety,
    suggested_dt_2D = cfl_safety * dxmin_for_cfl / vmax_for_cfl,
)

Δ = (Δspace..., cfl_info_preconstruction.suggested_dt_2D)
@show cfl_info_preconstruction Δ


# hereafter are the parameters which are important for modellers but maybe not interesting (so default numbers are given)

In [ ]:
orderBspace=1
orderBtime=1


pointsInSpace=3
pointsInTime=3

# the order of errors to be controlled
supplementaryOrder=2

# new parameters for interpolated Taylor expansion μ for field

fieldItpl = (ptsSpace = 1,ptsTime = 1,offsetSpace=1,offsetTime=1,YorderBspace=0,YorderBtime=0) #offsetSpace and offsetTime ∈ z 
# μ points should be distributed from y_min+offset*Δy to y_max-offset*Δy offset can be negative too


# new parameters for interpolated Taylor expansion μᶜ for material
materItpl = (ptsSpace = 1,ptsTime = 1,offsetSpace=1,offsetTime=1,YorderBspace=0,YorderBtime=0)

concreteParametersForOPTConstruction = @strdict famousEquationType Δ orderBtime orderBspace pointsInSpace pointsInTime supplementaryOrder fieldItpl materItpl

optRec = makeOPTsemiSymbolic(concreteParametersForOPTConstruction)
recette = optRec["recette"]
Ajiννᶜ = optRec["recette"].lhs.Ajiννᶜ

### treating model points based on timeMarching or not

In [ ]:
modelPoints = getModelPoints(models[1],pointsInTime,optRec["recette"].numbersOfTheSystem.numbersOfTheSystemL.timeMarching)#reference model dimension (this is important to explicitly define since the dimension/size of material variable array(s) can differ from the model domain)
tmpModel = (models=models, modelPoints=modelPoints, Δ=Δ)

In [ ]:
params = @strdict optRec=optRec modelFam=(tmpModel) absorbingBoundaries=nothing maskedRegionInSpace=nothing representation="matrixfree" 
numOpt = numericalOperatorConstruction(params)

In [ ]:
op = numOpt["numOperators"].numericalOperators.residual

## Source construction for numerical time marching

`timeMarchingSchemeLinear(...; sourceType="Explicit")` expects a source array with shape

```julia
(NforcePoints, NField, NtimeSource)
```

For the current numerical operator path, `NforcePoints == preparedLin.NpointsSpace`, so spatial localisation is encoded by zeros outside the source support. The helpers below create point sources, line/plane/hyperplane sources, and arbitrary distributed sources from masks or spatial profiles.


In [ ]:
function source_time_samples(Nt, Δt, timePointsUsedForOneStep; wavelet=nothing, t0=12Δt, f0=0.04)
    ntime = Nt + timePointsUsedForOneStep - 1
    t = (0:ntime-1) .* Δt
    if wavelet === nothing
        return Ricker.(t, t0, f0)
    else
        return wavelet.(t)
    end
end

function _normalise_source_weights!(w; normalise=:none)
    if normalise == :none || normalise === false
        return w
    elseif normalise == :sum
        s = sum(w)
        !iszero(s) && (w ./= s)
    elseif normalise == :l1
        s = sum(abs, w)
        !iszero(s) && (w ./= s)
    elseif normalise == :max
        s = maximum(abs, w)
        !iszero(s) && (w ./= s)
    else
        error("normalise should be :none, :sum, :l1, or :max")
    end
    return w
end

function make_sourceFull(preparedLin, spatialWeights, timeSignal; iField=1, amplitude=1.0)
    length(spatialWeights) == preparedLin.NforcePoints ||
        error("spatialWeights should have length preparedLin.NforcePoints=$(preparedLin.NforcePoints)")
    NForceField = hasproperty(preparedLin, :NForceField) ? preparedLin.NForceField : preparedLin.NField
    1 <= iField <= NForceField || error("iField should be between 1 and $(NForceField)")

    T = promote_type(eltype(spatialWeights), eltype(timeSignal), typeof(amplitude))
    sourceFull = zeros(T, preparedLin.NforcePoints, NForceField, length(timeSignal))
    for it in eachindex(timeSignal)
        sourceFull[:, iField, it] .= amplitude .* timeSignal[it] .* spatialWeights
    end
    return sourceFull
end

function point_source_weights(preparedLin, point::CartesianIndex; normalise=:none)
    weights = zeros(Float64, preparedLin.NforcePoints)
    LI = LinearIndices(preparedLin.spaceShape)
    weights[LI[point]] = 1.0
    return _normalise_source_weights!(weights; normalise)
end

function hyperplane_source_weights(preparedLin; fixed=Dict{Int,Int}(), profile=(I -> 1.0), normalise=:sum)
    weights = zeros(Float64, preparedLin.NforcePoints)
    LI = LinearIndices(preparedLin.spaceShape)
    for I in CartesianIndices(preparedLin.spaceShape)
        coords = Tuple(I)
        inside = all(coords[d] == v for (d, v) in fixed)
        if inside
            weights[LI[I]] = profile(I)
        end
    end
    return _normalise_source_weights!(weights; normalise)
end

function mask_source_weights(preparedLin, mask; profile=(I -> 1.0), normalise=:sum)
    weights = zeros(Float64, preparedLin.NforcePoints)
    LI = LinearIndices(preparedLin.spaceShape)

    if mask isa AbstractArray{Bool}
        size(mask) == preparedLin.spaceShape || error("Boolean mask size should be $(preparedLin.spaceShape)")
        for I in CartesianIndices(mask)
            mask[I] && (weights[LI[I]] = profile(I))
        end
    else
        for I in mask
            weights[LI[I]] = profile(I)
        end
    end

    return _normalise_source_weights!(weights; normalise)
end


### Prepare the numerical operator and source time function

Keep `maskedRegionInSpace=nothing` when you want a full-domain operator and a source distribution that is simply zero outside its support. If you later want to build a reduced operator on only selected source/test points, pass a collected vector of `CartesianIndex` values as `maskedRegionInSpace` during `numericalOperatorConstruction`.


In [ ]:
Nt = 200
Δnum = tmpModel.Δ
modelName = hasproperty(tmpModel, :modelName) ? tmpModel.modelName : "Marmousi_debug"

params = @strdict optRec=optRec modelFam=tmpModel absorbingBoundaries=nothing maskedRegionInSpace=nothing representation="matrixfree"
numOpt = numericalOperatorConstruction(params)
numOps = numOpt["numOperators"]
preparedLin = prepareLinearSystem(numOps)

timeSignal = source_time_samples(
    Nt,
    Δnum[end],
    preparedLin.timePointsUsedForOneStep;
    t0 = 12 * Δnum[end],
    f0 = (isdefined(@__MODULE__, :source_f0) ? source_f0 : (isdefined(@__MODULE__, :cfl_info) ? cfl_info.suggested_ricker_f0 : 0.04)),
)

@show preparedLin.spaceShape preparedLin.NforcePoints preparedLin.NField preparedLin.NForceField preparedLin.timePointsUsedForOneStep length(timeSignal)


### i) Point source

Choose the source point in the whole numerical grid coordinates. With no absorbing boundary this is the same as model-grid coordinates.


In [ ]:
sourcePoint = CartesianIndex(ntuple(d -> cld(preparedLin.spaceShape[d], 2), length(preparedLin.spaceShape))...)

w_point = point_source_weights(preparedLin, sourcePoint)
sourceFull_point = make_sourceFull(preparedLin, w_point, timeSignal; iField=(isdefined(@__MODULE__, :best_source_field) ? best_source_field : 1), amplitude=1.0)

@show sourcePoint size(sourceFull_point) sum(abs, sourceFull_point[:, 1, 1])


### ii) Line / plane source for plane-wave-like injection

Use `fixed` to pin one or more coordinates. In 2D, fixing one coordinate gives a line. In 3D, fixing one coordinate gives a plane, and fixing two coordinates gives a line.


In [ ]:
# Example for a 2D model: a horizontal line source at z = iz.
# For a 3D model, Dict(3 => iz) would be a horizontal plane source.
iz = max(1, min(preparedLin.spaceShape[end], 5))

w_plane = hyperplane_source_weights(
    preparedLin;
    fixed = Dict(length(preparedLin.spaceShape) => iz),
    profile = I -> 1.0,
    normalise = :sum,
)
sourceFull_plane = make_sourceFull(preparedLin, w_plane, timeSignal; iField=(isdefined(@__MODULE__, :best_source_field) ? best_source_field : 1), amplitude=1.0)

@show iz count(!iszero, w_plane) size(sourceFull_plane)


### iii) Distributed 1D / 2D / 3D / 4D source

A distributed source is just a spatial weight vector plus a time function. You can build it from a list of `CartesianIndex` values, a Boolean mask with `preparedLin.spaceShape`, or a custom spatial profile.


In [ ]:
# Rectangular / box-like distributed support around the centre.
centre = ntuple(d -> cld(preparedLin.spaceShape[d], 2), length(preparedLin.spaceShape))
radius = ntuple(d -> max(1, preparedLin.spaceShape[d] ÷ 20), length(preparedLin.spaceShape))

mask_distribution = CartesianIndex[]
for I in CartesianIndices(preparedLin.spaceShape)
    coords = Tuple(I)
    if all(abs(coords[d] - centre[d]) <= radius[d] for d in eachindex(coords))
        push!(mask_distribution, I)
    end
end

# A smooth compact-ish spatial profile on that mask.
profile(I) = exp(-sum(((Tuple(I)[d] - centre[d]) / radius[d])^2 for d in eachindex(centre)) / 2)

w_distribution = mask_source_weights(
    preparedLin,
    mask_distribution;
    profile = profile,
    normalise = :sum,
)
sourceFull_distribution = make_sourceFull(preparedLin, w_distribution, timeSignal; iField=(isdefined(@__MODULE__, :best_source_field) ? best_source_field : 1), amplitude=1.0)

@show length(mask_distribution) count(!iszero, w_distribution) size(sourceFull_distribution)


### Propagate one point source and make a Makie video

This cell keeps the wavefields in memory so you can quickly inspect and record them. Set `velocity_background = nothing` for wavefield-only videos, or pass a 2D velocity model with the same spatial shape as `preparedLin.spaceShape`.


In [ ]:
using LinearAlgebra, SparseArrays
using CairoMakie
CairoMakie.activate!()

function apply_forced_boundary_for_video!(A, b; leftValue=0.0, rightValue=0.0)
    A[1, :] .= 0
    A[1, 1] = 1
    b[1] = leftValue

    A[end, :] .= 0
    A[end, end] = 1
    b[end] = rightValue
    return A, b
end

function propagate_linear_frames(
    preparedLin,
    sourceFull,
    Nt;
    initialCondition=0.0,
    boundaryConditionForced=false,
    store_every=1,
    blowup_limit=Inf,
)
    NField = preparedLin.NField
    NForceField = hasproperty(preparedLin, :NForceField) ? preparedLin.NForceField : preparedLin.NField
    NpointsSpace = preparedLin.NpointsSpace
    timePointsUsedForOneStep = preparedLin.timePointsUsedForOneStep
    NknownTime = max(timePointsUsedForOneStep - 1, 0)

    size(sourceFull, 1) == preparedLin.NforcePoints || error("sourceFull first dimension mismatch")
    size(sourceFull, 2) == NForceField || error("sourceFull second dimension mismatch")
    size(sourceFull, 3) >= Nt + timePointsUsedForOneStep - 1 || error("sourceFull has too few time samples")

    knownField = fill(Float64(initialCondition), size(preparedLin.known_lhs_template))
    knownForce = fill(Float64(initialCondition), size(preparedLin.known_rhs_template))
    unknownField = fill(Float64(initialCondition), NpointsSpace, NField)

    A = sparse(preparedLin.A_template)
    b0 = copy(preparedLin.b_template)
    if boundaryConditionForced
        apply_forced_boundary_for_video!(A, b0; leftValue=0.0, rightValue=0.0)
    end
    factor = lu(A)
    b = copy(preparedLin.b_template)

    frames = Vector{Array{Float64}}()
    for it in 1:Nt
        knownForce .= sourceFull[:, :, it:it+timePointsUsedForOneStep-1]
        knownInputs = vcat(vec(knownField), vec(knownForce))
        preparedLin.b_fun!(b, knownInputs)
        if boundaryConditionForced
            b[1] = 0.0
            b[end] = 0.0
        end

        u = factor \ b
        unknownField .= reshape(real.(u), NpointsSpace, NField)

        if it % store_every == 0
            push!(frames, reshape(copy(unknownField[:, 1]), preparedLin.spaceShape...))
        end

        if NknownTime > 0
            if NknownTime > 1
                knownField[:, :, 1:end-1] .= knownField[:, :, 2:end]
            end
            knownField[:, :, end] .= unknownField
        end
    end
    return frames
end

function _finite_copy(A; fillvalue=0.0)
    B = Array{Float64}(undef, size(A))
    replaced = 0
    for I in eachindex(A)
        v = Float64(real(A[I]))
        if isfinite(v)
            B[I] = v
        else
            B[I] = fillvalue
            replaced += 1
        end
    end
    return B, replaced
end

function finite_frame_diagnostics(frames)
    total_bad = 0
    first_bad = nothing
    finite_max = 0.0
    for (i, frame) in pairs(frames)
        bad_here = count(!isfinite, frame)
        if bad_here > 0 && first_bad === nothing
            first_bad = i
        end
        total_bad += bad_here
        for v in frame
            if isfinite(v)
                finite_max = max(finite_max, abs(Float64(v)))
            end
        end
    end
    return (total_bad=total_bad, first_bad=first_bad, finite_max=finite_max)
end

function _background_for_plot(background, spaceShape)
    background === nothing && return nothing
    bg = Array(background)
    if size(bg) == spaceShape
        clean_bg, replaced = _finite_copy(bg)
    elseif length(bg) == prod(spaceShape)
        clean_bg, replaced = _finite_copy(reshape(bg, spaceShape...))
    else
        @warn "background size $(size(bg)) does not match wavefield spaceShape $spaceShape; ignoring background"
        return nothing
    end
    replaced > 0 && @warn "background had $replaced non-finite values; replaced by 0 for plotting"
    return clean_bg
end

function record_wave_video(
    frames;
    videoFile=joinpath(@__DIR__, "point_source_propagation.mp4"),
    background=nothing,
    sourcePoint=nothing,
    framerate=20,
    title="point source propagation",
    colormap=:balance,
    background_colormap=:grays,
    wave_alpha=0.72,
    clim=nothing,
)
    isempty(frames) && error("frames is empty")
    spaceShape = size(frames[1])
    bg = _background_for_plot(background, spaceShape)
    nd = length(spaceShape)

    diag = finite_frame_diagnostics(frames)
    if diag.total_bad > 0
        @warn "wavefield frames contain $(diag.total_bad) non-finite values; first affected stored frame is $(diag.first_bad). Replacing by 0 for plotting."
    end
    clean_frames = [first(_finite_copy(f)) for f in frames]

    if clim === nothing
        vmax = diag.finite_max
        clim = vmax == 0 ? 1.0 : vmax
    end

    fig = Figure(size=(900, 680))
    ax = Axis(fig[1, 1], aspect=DataAspect(), title=title)

    if nd == 1
        x = 1:spaceShape[1]
        yobs = Observable(vec(clean_frames[1]))
        lines!(ax, x, yobs, color=:dodgerblue3, linewidth=2)
        ylims!(ax, -clim, clim)
        if sourcePoint !== nothing
            vlines!(ax, [Tuple(sourcePoint)[1]], color=:red, linewidth=2)
        end
        record(fig, videoFile, eachindex(frames); framerate=framerate) do iframe
            yobs[] = vec(clean_frames[iframe])
            ax.title = "$title  |  frame $iframe/$(length(frames))"
        end
    elseif nd == 2
        if bg !== nothing
            heatmap!(ax, bg; colormap=background_colormap)
        end
        frame_obs = Observable(clean_frames[1])
        hm = heatmap!(ax, frame_obs; colormap=colormap, colorrange=(-clim, clim), alpha=bg === nothing ? 1.0 : wave_alpha)
        Colorbar(fig[1, 2], hm, label="wavefield")
        if sourcePoint !== nothing
            xy = Tuple(sourcePoint)
            scatter!(ax, [xy[1]], [xy[2]], color=:red, markersize=14)
        end
        record(fig, videoFile, eachindex(frames); framerate=framerate) do iframe
            frame_obs[] = clean_frames[iframe]
            ax.title = "$title  |  frame $iframe/$(length(frames))"
        end
    else
        error("Video helper currently supports 1D/2D frames. For 3D/4D, select a slice before recording.")
    end

    return videoFile
end


### One point-source run

The amplitude may need tuning depending on the equation scaling. Start modestly, inspect `maximum(abs, frames[...])`, then increase if the wavefield is too small.


In [ ]:
Nt = 180
store_every = 2
point_amplitude = 1.0

sourcePoint = CartesianIndex(ntuple(d -> cld(preparedLin.spaceShape[d], 2), length(preparedLin.spaceShape))...)
timeSignal = source_time_samples(
    Nt,
    Δnum[end],
    preparedLin.timePointsUsedForOneStep;
    t0 = 18 * Δnum[end],
    f0 = (isdefined(@__MODULE__, :source_f0) ? source_f0 : (isdefined(@__MODULE__, :cfl_info) ? cfl_info.suggested_ricker_f0 : 0.035)),
)

w_point = point_source_weights(preparedLin, sourcePoint)
sourceFull_point = make_sourceFull(preparedLin, w_point, timeSignal; iField=(isdefined(@__MODULE__, :best_source_field) ? best_source_field : 1), amplitude=point_amplitude)

frames_point = propagate_linear_frames(
    preparedLin,
    sourceFull_point,
    Nt;
    boundaryConditionForced=false,
    store_every=store_every,
)

@show length(frames_point) size(frames_point[1]) maximum(abs, frames_point[end])


### Record video with optional velocity background

For a 2D model, `velocity_background = models[1]` overlays the wavefield on the velocity model. Use `nothing` to see only the wavefield.


In [ ]:
velocity_background = length(preparedLin.spaceShape) == 2 ? models[1] : nothing
# velocity_background = nothing

videoFile = record_wave_video(
    frames_point;
    videoFile = joinpath(@__DIR__, "point_source_propagation.mp4"),
    background = velocity_background,
    sourcePoint = sourcePoint,
    framerate = 20,
    title = "point source in Marmousi background",
)

videoFile


If the video looks blank, first try increasing `point_amplitude` or lowering `clim` manually in `record_wave_video(...; clim=...)`. If it explodes, reduce `point_amplitude`, reduce `Δnum[end]`, or add absorbing boundaries before operator construction.


### CFL and motion diagnostics

For `2DacousticTime`, the model variable is velocity `v` in `∂t²u - v²Δu = f`. If `v*Δt/min(Δx,Δy)` is too large, the time marching will not look like a clean propagating wave. These helpers also plot frame-to-frame differences, which are often clearer than the raw field.


In [ ]:
using Statistics

function cfl_diagnostics(model_velocity, Δnum; cfl_safety=0.45, ppw=10, samples_per_period=20)
    v = vec(Float64.(model_velocity))
    finite_v = filter(isfinite, v)
    dt = Float64(Δnum[end])
    dx = Float64.(Δnum[1:end-1])
    dxmin = minimum(dx)
    dxmax = maximum(dx)
    vmax = maximum(abs, finite_v)
    vmin = minimum(abs, filter(!iszero, finite_v))
    cfl = vmax * dt / dxmin

    # MKS units: velocity [m/s], dx [m], dt [s], frequency [Hz].
    # For a Ricker wavelet in flexOPT, f0 is the dominant/peak frequency.
    fmax_spatial = vmin / (ppw * dxmax)
    fmax_temporal = 1 / (samples_per_period * dt)
    suggested_ricker_f0 = min(fmax_spatial, fmax_temporal)

    return (
        v_min=minimum(finite_v),
        v_median=median(finite_v),
        v_max=maximum(finite_v),
        dt=dt,
        dxmin=dxmin,
        dxmax=dxmax,
        cfl_max=cfl,
        suggested_dt_2D=cfl_safety * dxmin / vmax,
        ppw=ppw,
        samples_per_period=samples_per_period,
        fmax_spatial=fmax_spatial,
        fmax_temporal=fmax_temporal,
        suggested_ricker_f0=suggested_ricker_f0,
        suggested_ricker_t0=1.5 / suggested_ricker_f0,
    )
end

function ricker_time_signal(Nt, Δt; f0, t0=1.5/f0, timePointsUsedForOneStep=1)
    ntime = Nt + timePointsUsedForOneStep - 1
    t = (0:ntime-1) .* Δt
    return t, Ricker.(t, t0, f0)
end

function dominant_frequency_scan(signal, Δt; nfreq=512)
    y = Float64.(signal) .- mean(Float64.(signal))
    n = length(y)
    duration = n * Δt
    fmin = 1 / duration
    fmax = 0.5 / Δt
    freqs = range(fmin, fmax; length=nfreq)
    t = (0:n-1) .* Δt
    spectrum = [abs(sum(y .* exp.(-2im * π * f .* t))) for f in freqs]
    imax = argmax(spectrum)
    return (f_peak=freqs[imax], peak_amplitude=spectrum[imax], frequencies=freqs, spectrum=spectrum)
end

function ricker_diagnostics(model_velocity, Δnum; f0=nothing, Nt=200, timePointsUsedForOneStep=1, ppw=10, samples_per_period=20, t0=nothing)
    cfl = cfl_diagnostics(model_velocity, Δnum; ppw=ppw, samples_per_period=samples_per_period)
    f0_used = f0 === nothing ? cfl.suggested_ricker_f0 : Float64(f0)
    t0_used = t0 === nothing ? 1.5 / f0_used : Float64(t0)
    t, signal = ricker_time_signal(Nt, cfl.dt; f0=f0_used, t0=t0_used, timePointsUsedForOneStep=timePointsUsedForOneStep)
    freqinfo = dominant_frequency_scan(signal, cfl.dt)
    return (
        f0=f0_used,
        t0=t0_used,
        duration=t[end],
        dominant_frequency_scan=freqinfo.f_peak,
        wavelength_min_velocity=cfl.v_min / f0_used,
        points_per_wavelength_min_velocity=(cfl.v_min / f0_used) / cfl.dxmax,
        samples_per_period=1 / (f0_used * cfl.dt),
        max_abs=maximum(abs, signal),
        first_peak_time=t[argmax(abs.(signal))],
        cfl=cfl,
    )
end

function frame_motion_report(frames)
    rows = NamedTuple[]
    previous = nothing
    for (i, frame) in pairs(frames)
        _, nbad, fmax = _finite_plot_matrix(frame)
        dnorm = previous === nothing ? 0.0 : norm(frame .- previous)
        dmax = previous === nothing ? 0.0 : maximum(abs, frame .- previous)
        push!(rows, (frame=i, nbad=nbad, maxabs=fmax, norm=norm(frame), diff_norm=dnorm, diff_max=dmax))
        previous = frame
    end
    return rows
end

function difference_frames(frames)
    length(frames) >= 2 || error("need at least two frames")
    return [frames[i] .- frames[i-1] for i in 2:length(frames)]
end


In [ ]:
cfl_info = cfl_diagnostics(models[1], Δnum)
ricker_info = ricker_diagnostics(
    models[1],
    Δnum;
    Nt=Nt,
    timePointsUsedForOneStep=preparedLin.timePointsUsedForOneStep,
)
source_f0 = ricker_info.f0
source_t0 = ricker_info.t0
@show cfl_info
@show ricker_info
@show source_f0 source_t0
source_f0 = ricker_info.f0
source_t0 = ricker_info.t0
f0 = source_f0
t0 = source_t0
# MKS reminder:
# Δspace is in meters, Δt is in seconds, velocities are in m/s, and Ricker f0 is in Hz.
# If propagation looks too smooth/static, try f0 close to cfl_info.suggested_ricker_f0.


### Numerical propagation diagnostics

If no wave propagates, first check whether the numerical system sees the source. These diagnostics do not plot; they report norms and the best source field/component to use.


In [ ]:
function _sparse_norms(A)
    return (
        size=size(A),
        nnz=nnz(A),
        maxabs=nnz(A) == 0 ? 0.0 : maximum(abs, nonzeros(A)),
        norm=norm(A),
    )
end

function diagnose_prepared_linear_system(preparedLin)
    println("spaceShape = ", preparedLin.spaceShape)
    println("NpointsSpace = ", preparedLin.NpointsSpace)
    println("NField = ", preparedLin.NField)
    println("NForceField = ", hasproperty(preparedLin, :NForceField) ? preparedLin.NForceField : preparedLin.NField)
    println("timePointsUsedForOneStep = ", preparedLin.timePointsUsedForOneStep)
    println("A_unknown: ", _sparse_norms(preparedLin.A_unknown))
    println("L_known:   ", _sparse_norms(preparedLin.L_known))
    println("R_force:   ", _sparse_norms(preparedLin.R_force))
    if hasproperty(preparedLin, :right_operator)
        println("right_operator.size = ", preparedLin.right_operator.size)
        println("right_operator.nnz = ", length(preparedLin.right_operator.table.vals))
    end
    row_nnz_A = vec(sum(abs.(preparedLin.A_unknown) .> 0; dims=2))
    row_nnz_R = vec(sum(abs.(preparedLin.R_force) .> 0; dims=2))
    println("rows with A entries = ", count(!iszero, row_nnz_A), " / ", length(row_nnz_A))
    println("rows with R entries = ", count(!iszero, row_nnz_R), " / ", length(row_nnz_R))
    return nothing
end

function source_field_response(preparedLin, sourcePoint, timeSignal; amplitude=1.0, it=nothing)
    LI = LinearIndices(preparedLin.spaceShape)
    src_linear = LI[sourcePoint]
    last_start = length(timeSignal) - preparedLin.timePointsUsedForOneStep + 1
    last_start >= 1 || error("timeSignal is too short for timePointsUsedForOneStep=$(preparedLin.timePointsUsedForOneStep)")
    if it === nothing
        it = min(argmax(abs.(timeSignal)), last_start)
    else
        1 <= it <= last_start || error("it should be in 1:$last_start for this timeSignal")
    end

    out = NamedTuple[]
    NForceField = hasproperty(preparedLin, :NForceField) ? preparedLin.NForceField : preparedLin.NField
    for iField in 1:NForceField
        w = zeros(Float64, preparedLin.NforcePoints)
        w[src_linear] = 1.0
        sourceFull = make_sourceFull(preparedLin, w, timeSignal; iField=iField, amplitude=amplitude)
        knownForce = sourceFull[:, :, it:it+preparedLin.timePointsUsedForOneStep-1]
        knownInputs = vcat(vec(zero(preparedLin.known_lhs_template)), vec(knownForce))
        b = copy(preparedLin.b_template)
        preparedLin.b_fun!(b, knownInputs)
        push!(out, (
            iField=iField,
            time_index=it,
            source_norm=norm(knownForce),
            b_norm=norm(b),
            b_maximum=maximum(abs, b),
            b_nonzero=count(!iszero, b),
        ))
    end
    return out
end

function one_step_response(preparedLin, sourceFull; it=1, boundaryConditionForced=false)
    knownField = zero(preparedLin.known_lhs_template)
    knownForce = sourceFull[:, :, it:it+preparedLin.timePointsUsedForOneStep-1]
    knownInputs = vcat(vec(knownField), vec(knownForce))
    b = copy(preparedLin.b_template)
    preparedLin.b_fun!(b, knownInputs)

    A = sparse(preparedLin.A_template)
    if boundaryConditionForced
        apply_forced_boundary_for_video!(A, b; leftValue=0.0, rightValue=0.0)
    end
    u = A \ b
    ufield = reshape(real.(u), preparedLin.NpointsSpace, preparedLin.NField)
    return (
        b_norm=norm(b),
        b_maximum=maximum(abs, b),
        b_nonzero=count(!iszero, b),
        u_norm=norm(u),
        u_maximum=maximum(abs, u),
        u_nonzero=count(!iszero, u),
        ufield=ufield,
    )
end


function _matrix_row_abs_sums(A)
    return vec(sum(abs.(A); dims=2))
end

function operator_growth_diagnostics(preparedLin)
    A = sparse(preparedLin.A_unknown)
    L = sparse(preparedLin.L_known)
    R = sparse(preparedLin.R_force)
    diagA = abs.(diag(A))
    rowA = _matrix_row_abs_sums(A)
    offdiagA = rowA .- diagA
    rowL = _matrix_row_abs_sums(L)
    rowR = _matrix_row_abs_sums(R)
    return (
        A_size=size(A),
        A_nnz=nnz(A),
        A_diag_min=isempty(diagA) ? 0.0 : minimum(diagA),
        A_diag_median=isempty(diagA) ? 0.0 : median(diagA),
        A_diag_max=isempty(diagA) ? 0.0 : maximum(diagA),
        A_offdiag_over_diag_max=maximum(offdiagA ./ max.(diagA, eps(Float64))),
        L_row_sum_max=maximum(rowL),
        R_row_sum_max=maximum(rowR),
        R_over_A_diag_max=maximum(rowR ./ max.(diagA, eps(Float64))),
        condest_dense_sample=size(A,1) <= 5000 ? cond(Matrix(A)) : missing,
    )
end

function homogeneous_zero_test(preparedLin; Nt=20, store_every=1)
    sourceFull_zero = zeros(eltype(preparedLin.known_rhs_template), preparedLin.NforcePoints, preparedLin.NForceField, Nt + preparedLin.timePointsUsedForOneStep - 1)
    frames_zero = propagate_linear_frames(preparedLin, sourceFull_zero, Nt; store_every=store_every, blowup_limit=1e12)
    return wavefield_snapshot_report(frames_zero)
end

function source_amplitude_sweep(preparedLin, sourcePoint, timeSignal; amplitudes=(1e-12, 1e-10, 1e-8, 1e-6), iField=1, Nt=20)
    rows = NamedTuple[]
    w = point_source_weights(preparedLin, sourcePoint)
    for amp in amplitudes
        sourceFull = make_sourceFull(preparedLin, w, timeSignal; iField=iField, amplitude=amp)
        frames = propagate_linear_frames(preparedLin, sourceFull, Nt; store_every=max(1, Nt ÷ 5), blowup_limit=1e30)
        report = wavefield_snapshot_report(frames)
        push!(rows, (amplitude=amp, last=report[end], nframes=length(frames)))
    end
    return rows
end


function _distance_from_source_table(frame, sourcePoint; threshold_fraction=1e-6)
    f = abs.(Float64.(frame))
    fmax = maximum(f)
    threshold = threshold_fraction * max(fmax, eps(Float64))
    active = findall(>(threshold), f)
    if isempty(active)
        return (nactive=0, max_distance=0.0, mean_distance=0.0, fmax=fmax)
    end
    distances = [sqrt(sum((Tuple(I) .- Tuple(sourcePoint)).^2)) for I in active]
    return (
        nactive=length(active),
        max_distance=maximum(distances),
        mean_distance=mean(distances),
        fmax=fmax,
    )
end

function propagation_radius_report(frames, sourcePoint; threshold_fraction=1e-6)
    return [(frame=i, _distance_from_source_table(frame, sourcePoint; threshold_fraction=threshold_fraction)...) for (i, frame) in pairs(frames)]
end

function impulse_sourceFull(preparedLin, sourcePoint; iField=1, amplitude=1.0, Nt=20, impulse_time=1)
    timeSignal = zeros(Float64, Nt + preparedLin.timePointsUsedForOneStep - 1)
    timeSignal[impulse_time] = 1.0
    w = point_source_weights(preparedLin, sourcePoint)
    return make_sourceFull(preparedLin, w, timeSignal; iField=iField, amplitude=amplitude)
end

function impulse_propagation_test(preparedLin, sourcePoint; iField=1, amplitude=1.0, Nt=20, store_every=1, threshold_fraction=1e-8)
    sourceFull = impulse_sourceFull(preparedLin, sourcePoint; iField=iField, amplitude=amplitude, Nt=Nt)
    frames = propagate_linear_frames(preparedLin, sourceFull, Nt; store_every=store_every, blowup_limit=1e20)
    return (
        frames=frames,
        snapshots=wavefield_snapshot_report(frames),
        radius=propagation_radius_report(frames, sourcePoint; threshold_fraction=threshold_fraction),
    )
end


function frame_extrema_report(frames, sourcePoint)
    rows = NamedTuple[]
    for (i, frame) in pairs(frames)
        A = Float64.(frame)
        absA = abs.(A)
        imax = argmax(absA)
        maxpoint = CartesianIndices(size(A))[imax]
        distance = sqrt(sum((Tuple(maxpoint) .- Tuple(sourcePoint)).^2))
        push!(rows, (
            frame=i,
            maxpoint=maxpoint,
            distance_from_source=distance,
            value=A[imax],
            maxabs=absA[imax],
            minimum=minimum(A),
            maximum=maximum(A),
        ))
    end
    return rows
end


In [ ]:
diagnose_prepared_linear_system(preparedLin)

sourcePoint_test = CartesianIndex(ntuple(d -> cld(preparedLin.spaceShape[d], 2), length(preparedLin.spaceShape))...)
timeSignal_test = source_time_samples(
    50,
    Δnum[end],
    preparedLin.timePointsUsedForOneStep;
    t0 = 8 * Δnum[end],
    f0 = (isdefined(@__MODULE__, :source_f0) ? source_f0 : (isdefined(@__MODULE__, :cfl_info) ? cfl_info.suggested_ricker_f0 : 0.035)),
)

field_responses = source_field_response(
    preparedLin,
    sourcePoint_test,
    timeSignal_test;
    amplitude=1.0,
)
field_responses


In [ ]:
growth_diag = operator_growth_diagnostics(preparedLin)
@show growth_diag

zero_report = homogeneous_zero_test(preparedLin; Nt=20, store_every=5)
@show zero_report[1] zero_report[end]


In [ ]:
if !isdefined(@__MODULE__, :frame_extrema_report)
    function frame_extrema_report(frames, sourcePoint)
        rows = NamedTuple[]
        for (i, frame) in pairs(frames)
            A = Float64.(frame)
            absA = abs.(A)
            imax = argmax(absA)
            maxpoint = CartesianIndices(size(A))[imax]
            distance = sqrt(sum((Tuple(maxpoint) .- Tuple(sourcePoint)).^2))
            push!(rows, (
                frame=i,
                maxpoint=maxpoint,
                distance_from_source=distance,
                value=A[imax],
                maxabs=absA[imax],
                minimum=minimum(A),
                maximum=maximum(A),
            ))
        end
        return rows
    end
end

impulse_test = impulse_propagation_test(
    preparedLin,
    sourcePoint_debug;
    iField=(isdefined(@__MODULE__, :best_source_field) ? best_source_field : 1),
    amplitude=1e-12,
    Nt=20,
    store_every=1,
    threshold_fraction=1e-8,
)

@show impulse_test.snapshots[1] impulse_test.snapshots[end]
impulse_extrema = frame_extrema_report(impulse_test.frames, sourcePoint_debug)
@show impulse_extrema[1] impulse_extrema[end]
impulse_test.radius


In [ ]:
vmed = median(vec(Float64.(models[1])))
expected_cells = vmed * (20 * Δnum[end]) / minimum(Δnum[1:end-1])
@show vmed Δnum[end] expected_cells

In [ ]:
impulse_test.radius = propagation_radius_report(
    impulse_test.frames,
    sourcePoint_debug;
    threshold_fraction=1e-12,
)
impulse_test.radius

Choose the field/component with the largest `b_norm`. If all `b_norm` values are zero, the RHS/source operator is not connected to the source array yet.


In [ ]:
best_source_field = argmax(getproperty.(field_responses, :b_norm))
@show best_source_field field_responses[best_source_field]

w_test = point_source_weights(preparedLin, sourcePoint_test)
sourceFull_test = make_sourceFull(preparedLin, w_test, timeSignal_test; iField=best_source_field, amplitude=1.0)
step_test = one_step_response(preparedLin, sourceFull_test; it=argmax(abs.(timeSignal_test)))

@show step_test.b_norm step_test.b_maximum step_test.b_nonzero step_test.u_norm step_test.u_maximum step_test.u_nonzero


If `u_nonzero` is only one or a few grid points, then the solve is injecting but not coupling spatially. That means the left operator split or the chosen OPT recipe is not producing spatial propagation. If `u_nonzero` is large, use `best_source_field` in the propagation cells below.


### Snapshot diagnostics before making a video

Use this before recording. It plots a few selected stored frames, sanitizes `NaN`/`Inf` values for display, and writes the number of non-finite values in each panel title. This is the fastest way to see whether the simulation is stable.


In [ ]:
function _finite_plot_matrix(A; fillvalue=0.0)
    B = Array{Float64}(undef, size(A))
    nbad = 0
    finite_max = 0.0
    for I in eachindex(A)
        v = Float64(real(A[I]))
        if isfinite(v)
            B[I] = v
            finite_max = max(finite_max, abs(v))
        else
            B[I] = fillvalue
            nbad += 1
        end
    end
    return B, nbad, finite_max
end

function wavefield_snapshot_report(frames)
    rows = NamedTuple[]
    for (i, frame) in pairs(frames)
        _, nbad, finite_max = _finite_plot_matrix(frame)
        push!(rows, (frame=i, nbad=nbad, finite_max=finite_max, minimum=minimum(skipmissing(vec(ifelse.(isfinite.(frame), frame, missing)))), maximum=maximum(skipmissing(vec(ifelse.(isfinite.(frame), frame, missing))))))
    end
    return rows
end


function _snapshot_background_for_plot(background, spaceShape)
    background === nothing && return nothing
    bg = Array(background)
    if size(bg) == spaceShape
        B, replaced, _ = _finite_plot_matrix(bg)
    elseif length(bg) == prod(spaceShape)
        B, replaced, _ = _finite_plot_matrix(reshape(bg, spaceShape...))
    else
        @warn "background size $(size(bg)) does not match wavefield spaceShape $spaceShape; ignoring background"
        return nothing
    end
    replaced > 0 && @warn "background had $replaced non-finite values; replaced by 0 for plotting"
    return B
end

function plot_wave_snapshots(
    frames;
    indices=nothing,
    background=nothing,
    sourcePoint=nothing,
    clim=nothing,
    ncols=3,
    title="wavefield snapshots",
    colormap=:balance,
    background_colormap=:grays,
    wave_alpha=0.72,
)
    isempty(frames) && error("frames is empty")
    if indices === nothing
        indices = unique(round.(Int, range(1, length(frames), length=min(9, length(frames)))))
    end
    indices = collect(indices)

    clean_frames = Array{Float64}[]
    nbad = Int[]
    finite_maxima = Float64[]
    for i in indices
        B, bad, fmax = _finite_plot_matrix(frames[i])
        push!(clean_frames, B)
        push!(nbad, bad)
        push!(finite_maxima, fmax)
    end

    if clim === nothing
        vmax = maximum(finite_maxima)
        clim = vmax == 0 ? 1.0 : vmax
    end

    bg = _snapshot_background_for_plot(background, size(frames[1]))
    n = length(indices)
    nrows = cld(n, ncols)
    fig = Figure(size=(320*ncols, 300*nrows))

    for (k, iframe) in enumerate(indices)
        r = cld(k, ncols)
        c = k - (r - 1) * ncols
        ax = Axis(fig[r, c], aspect=DataAspect(), title="frame $iframe | bad=$(nbad[k])")
        if length(size(frames[1])) == 1
            lines!(ax, vec(clean_frames[k]), color=:dodgerblue3, linewidth=2)
            ylims!(ax, -clim, clim)
            if sourcePoint !== nothing
                vlines!(ax, [Tuple(sourcePoint)[1]], color=:red, linewidth=2)
            end
        elseif length(size(frames[1])) == 2
            if bg !== nothing
                heatmap!(ax, bg; colormap=background_colormap)
            end
            heatmap!(ax, clean_frames[k]; colormap=colormap, colorrange=(-clim, clim), alpha=bg === nothing ? 1.0 : wave_alpha)
            if sourcePoint !== nothing
                xy = Tuple(sourcePoint)
                scatter!(ax, [xy[1]], [xy[2]], color=:red, markersize=10)
            end
        else
            error("Snapshot helper supports 1D/2D frames. Select a slice first for higher dimensions.")
        end
    end

    Label(fig[0, :], title)
    return fig
end


Run a shorter, safer point-source test first. If this already produces non-finite frames, reduce `point_amplitude` or `Nt`, or try `boundaryConditionForced=true`.


In [ ]:
Nt_debug =50
store_every_debug = 10
point_amplitude_debug = 1e-6

sourcePoint_debug = CartesianIndex(ntuple(d -> cld(preparedLin.spaceShape[d], 2), length(preparedLin.spaceShape))...)
timeSignal_debug = source_time_samples(
    Nt_debug,
    Δnum[end],
    preparedLin.timePointsUsedForOneStep;
    t0 = 8 * Δnum[end],
    f0 = (isdefined(@__MODULE__, :source_f0) ? source_f0 : (isdefined(@__MODULE__, :cfl_info) ? cfl_info.suggested_ricker_f0 : 0.035)),
)

w_point_debug = point_source_weights(preparedLin, sourcePoint_debug)
sourceFull_debug = make_sourceFull(preparedLin, w_point_debug, timeSignal_debug; iField=(isdefined(@__MODULE__, :best_source_field) ? best_source_field : 1), amplitude=point_amplitude_debug)

frames_debug = propagate_linear_frames(
    preparedLin,
    sourceFull_debug,
    Nt_debug;
    boundaryConditionForced=false,
    store_every=store_every_debug,
)

report_debug = wavefield_snapshot_report(frames_debug)
@show length(frames_debug) report_debug[1] report_debug[end]


In [ ]:
velocity_background_debug = length(preparedLin.spaceShape) == 2 ? models[1] : nothing
# velocity_background_debug = nothing

snapshot_indices = unique(round.(Int, range(1, length(frames_debug), length=min(9, length(frames_debug)))))
fig_snapshots = plot_wave_snapshots(
    frames_debug;
    indices=snapshot_indices,
    background=velocity_background_debug,
    sourcePoint=sourcePoint_debug,
    title="debug point-source snapshots",
)
fig_snapshots


In [ ]:
fig_snapshots

If `bad=0` in the snapshot titles and the panels look reasonable, then use the same amplitude/time settings for `record_wave_video`. If `bad>0`, the propagation is unstable before plotting, so the next fix is numerical stability rather than Makie.


### Write snapshots directly to PDF

If Makie/Jupyter stops displaying `fig_snapshots`, write the figure directly to disk. This avoids keeping a large rendered figure in the notebook output.


In [ ]:
function save_wave_snapshot_pdf(
    frames;
    filename=joinpath(@__DIR__, "wave_snapshots.pdf"),
    indices=nothing,
    background=nothing,
    sourcePoint=nothing,
    clim=nothing,
    ncols=3,
    title="wavefield snapshots",
    kwargs...,
)
    fig = plot_wave_snapshots(
        frames;
        indices=indices,
        background=background,
        sourcePoint=sourcePoint,
        clim=clim,
        ncols=ncols,
        title=title,
        kwargs...,
    )
    save(filename, fig)
    empty!(fig.scene)  # release most displayed scene resources
    GC.gc()
    return filename
end


In [ ]:
snapshot_indices = unique(round.(Int, range(1, length(frames_debug), length=min(12, length(frames_debug)))))

snapshot_pdf = save_wave_snapshot_pdf(
    frames_debug;
    filename=joinpath(@__DIR__, "point_source_debug_snapshots.pdf"),
    indices=snapshot_indices,
    background=velocity_background_debug,
    sourcePoint=sourcePoint_debug,
    title="debug point-source snapshots",
)

snapshot_pdf


### Plot motion instead of raw amplitude

If raw snapshots look static, plot `frame[i] - frame[i-1]`. This emphasizes propagation/motion and removes stationary components.


In [ ]:
motion_debug = frame_motion_report(frames_debug)
@show motion_debug[1] motion_debug[end]

frames_debug_diff = difference_frames(frames_debug)
diff_indices = unique(round.(Int, range(1, length(frames_debug_diff), length=min(12, length(frames_debug_diff)))))

snapshot_diff_pdf = save_wave_snapshot_pdf(
    frames_debug_diff;
    filename=joinpath(@__DIR__, "point_source_debug_difference_snapshots.pdf"),
    indices=diff_indices,
    background=nothing,
    sourcePoint=sourcePoint_debug,
    title="debug point-source frame differences",
)

snapshot_diff_pdf


### Inspect coefficients at one point

Use this to verify the local numerical stencil. For one equation at one grid point, it lists every coefficient by field, time slot, neighbour point, and spatial offset. Use `which=:left` for the unknown field operator and `which=:right` for the source-distribution operator coming from `optRec["recette"].rhs.Γjiννᶜ`. The last time slot is the future unknown for the LHS, and the corresponding source time sample for the RHS.


In [ ]:
function _get_numop(numOps, which::Symbol)
    ops = numOps isa AbstractDict ? numOps["numericalOperators"] : numOps.numericalOperators
    return getproperty(ops, which)
end

function _point_linear_index(op, point)
    νWhole = op.geometry.νWhole
    if point isa Integer
        1 <= point <= length(νWhole) || error("point integer should be in 1:$(length(νWhole))")
        return Int(point), νWhole[Int(point)]
    elseif point isa CartesianIndex
        point in νWhole || error("point $point is not in op.geometry.νWhole")
        pointLinear = LinearIndices(νWhole)[point]
        return Int(pointLinear), point
    else
        error("point should be an Int linear point index or a CartesianIndex")
    end
end

function _time_role(iT, activeTimePoints)
    if iT == activeTimePoints
        return :future_unknown
    elseif iT == activeTimePoints - 1
        return :present_known
    else
        return Symbol("past_known_tminus$(activeTimePoints - iT)")
    end
end

function coefficient_rows_at_point(op, point; iExpr=1, atol=0.0)
    pointLinear, pointCI = _point_linear_index(op, point)
    nPoints = length(op.geometry.νWhole)
    activeTimePoints = op.geometry.activeTimePoints
    spaceShape = Tuple(op.geometry.wholeRegionPointsSpace)
    Nexpr = op.size[1] ÷ nPoints
    Nfield = op.size[2] ÷ (nPoints * activeTimePoints)

    1 <= iExpr <= Nexpr || error("iExpr should be in 1:$Nexpr")

    residualLI = LinearIndices((Nexpr, nPoints))
    targetRow = residualLI[iExpr, pointLinear]
    fieldTimeSpaceCI = CartesianIndices((Nfield, activeTimePoints, spaceShape...))

    rows = NamedTuple[]
    for k in eachindex(op.table.vals)
        Int(op.table.rows[k]) == targetRow || continue
        val = op.table.vals[k]
        abs(val) > atol || continue

        ci = fieldTimeSpaceCI[Int(op.table.cols[k])]
        iField = ci[1]
        iT = ci[2]
        neighbour = CartesianIndex(Tuple(ci)[3:end])
        offset = Tuple(neighbour - pointCI)
        timeRole = _time_role(iT, activeTimePoints)

        push!(rows, (
            k = k,
            residual_row = targetRow,
            expr = iExpr,
            point = pointCI,
            field = iField,
            time_slot = iT,
            time_role = timeRole,
            neighbour = neighbour,
            offset = offset,
            coef = val,
            abscoef = abs(val),
        ))
    end

    sort!(rows, by = r -> (r.time_slot, r.field, r.offset))
    return rows
end

function inspect_operator_coefficients_at_point(numOps, point; which=:residual, iExpr=1, atol=0.0)
    op = _get_numop(numOps, which)
    rows = coefficient_rows_at_point(op, point; iExpr=iExpr, atol=atol)
    println("operator = ", which)
    println("point = ", point, ", iExpr = ", iExpr, ", ncoef = ", length(rows))
    for r in rows
        println(
            "t=", r.time_slot,
            " (", r.time_role, ")",
            " field=", r.field,
            " offset=", r.offset,
            " neighbour=", r.neighbour,
            " coef=", r.coef,
        )
    end
    return rows
end

function coefficient_table_by_time(rows)
    slots = sort(unique(getproperty.(rows, :time_slot)))
    return [(
        time_slot=s,
        time_role=first(r.time_role for r in rows if r.time_slot == s),
        n=length(filter(r -> r.time_slot == s, rows)),
        sumcoef=sum(r.coef for r in rows if r.time_slot == s),
        sumabs=sum(r.abscoef for r in rows if r.time_slot == s),
    ) for s in slots]
end

function coefficient_rows_for_role(rows, role::Symbol)
    return filter(r -> r.time_role == role, rows)
end


In [ ]:
# Example: inspect the centre point of the RHS/source-distribution operator.
inspectPoint = CartesianIndex(ntuple(d -> cld(preparedLin.spaceShape[d], 2), length(preparedLin.spaceShape))...)
coef_rows_right = inspect_operator_coefficients_at_point(numOps, inspectPoint; which=:right, iExpr=1, atol=0.0)
coefficient_table_by_time(coef_rows_right)


In [ ]:
# Compare LHS and RHS separately at the same point.
coef_rows_left = inspect_operator_coefficients_at_point(numOps, inspectPoint; which=:left, iExpr=1, atol=0.0)
coef_rows_right = inspect_operator_coefficients_at_point(numOps, inspectPoint; which=:right, iExpr=1, atol=0.0)

(
    left_by_time = coefficient_table_by_time(coef_rows_left),
    right_by_time = coefficient_table_by_time(coef_rows_right),
)
